# 1. Setup & Imports
This section imports required libraries, configures paths, and defines the experiment scope for FinSight LightGBM forecasting.

In [1]:
import sys
sys.path.append("../../../")

import json
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
from lightgbm import LGBMRegressor

from backend.data.data_loader import fetch_stock_data
from backend.data.preprocessor import FinSightPreprocessor

optuna.logging.set_verbosity(optuna.logging.INFO)

TICKER = "SBIN.NS"
START = "2020-01-01"
END = "2026-05-01"
TEST_START = "2025-01-01"

safe_ticker = ''.join(ch if ch.isalnum() else '_' for ch in TICKER)
ARTIFACTS_DIR = Path("./")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker: {TICKER}")
print(f"Date range: {START} to {END}")
print(f"Artifacts dir: {ARTIFACTS_DIR.resolve()}")

c:\Users\gaura\Desktop\FinSight\finsightvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ticker: SBIN.NS
Date range: 2020-01-01 to 2026-05-01
Artifacts dir: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm


# 2. Load Data
Load OHLCV data using FinSight's shared data loader and inspect shape and schema.

In [2]:
df = fetch_stock_data(TICKER, START, END)
print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (1567, 9)


Price,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread
Date,,,,,,,,,
2020-01-02,334.500000,339.850006,333.350006,339.299988,20324236,307.534393,0.014501,0.014397,6.500000
2020-01-03,337.950012,337.950012,332.000000,333.700012,21853208,302.458710,-0.016504,-0.016642,5.950012
2020-01-06,331.700012,331.700012,317.700012,319.000000,35645325,289.134888,-0.044052,-0.045051,14.000000
2020-01-07,324.450012,327.000000,315.399994,318.399994,50966826,288.591095,-0.001881,-0.001883,11.600006
2020-01-08,312.100006,321.500000,311.000000,319.799988,44527485,289.860016,0.004397,0.004387,10.500000


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1567 entries, 2020-01-02 to 2026-05-01
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1567 non-null   float64
 1   High          1567 non-null   float64
 2   Low           1567 non-null   float64
 3   Close         1567 non-null   float64
 4   Volume        1567 non-null   int64  
 5   Adj Close     1567 non-null   float64
 6   daily_return  1567 non-null   float64
 7   log_return    1567 non-null   float64
 8   hl_spread     1567 non-null   float64
dtypes: float64(8), int64(1)
memory usage: 122.4 KB


# 3. Feature Engineering
Apply FinSight preprocessing and optionally enrich the feature frame using graph features if the JSON artifact exists.
The target variable is next-day Close (`Close.shift(-1)`).

In [3]:
preprocessor = FinSightPreprocessor(ticker=TICKER)
processed_df = preprocessor.fit_transform(df)

graph_features_path = ARTIFACTS_DIR / "graph_features.json"
if graph_features_path.exists():
    with graph_features_path.open("r", encoding="utf-8") as f:
        graph_features = json.load(f)
    for k, v in graph_features.items():
        processed_df[k] = float(v)
    print(f"Loaded graph features from {graph_features_path}")
else:
    graph_features = {}
    print(f"No graph features found at {graph_features_path}; continuing without them.")

model_df = processed_df.copy()
model_df["target"] = model_df["Close"].shift(-1)
model_df = model_df.dropna(subset=["target"]).replace([np.inf, -np.inf], np.nan).dropna()

print("Final model frame shape:", model_df.shape)
display(model_df.head())
print("Feature columns (excluding target):", len([c for c in model_df.columns if c != "target"]))

No graph features found at graph_features.json; continuing without them.
Final model frame shape: (1547, 15)


,Open,High,Low,Close,Volume,Adj Close,daily_return,log_return,hl_spread,Close_lag_1,Close_lag_5,Close_lag_10,rolling_mean_20,rolling_std_20,target
Date,,,,,,,,,,,,,,,
2020-01-29,-1.123457,-1.133921,-1.112527,-1.127601,-0.060100,-1.119131,0.168870,0.177542,-1.022843,-1.132165,-1.124108,-1.085850,-1.095752,-0.861506,-1.151450
2020-01-30,-1.128020,-1.146083,-1.153875,-1.151450,0.431931,-1.140636,-0.979159,-0.973986,-0.202971,-1.126561,-1.094723,-1.089832,-1.101759,-0.920186,-1.119306
2020-01-31,-1.140671,-1.125676,-1.141554,-1.119306,2.785020,-1.111651,1.225108,1.214481,0.032964,-1.150431,-1.091180,-1.112048,-1.104963,-0.979974,-1.203710
2020-02-03,-1.185054,-1.186897,-1.196893,-1.203710,1.270363,-1.187760,-3.317324,-3.403017,-0.155785,-1.118259,-1.123900,-1.128814,-1.109353,-0.809970,-1.169907
2020-02-04,-1.185469,-1.184012,-1.189584,-1.169907,1.066792,-1.157279,1.347613,1.333393,-0.279653,-1.202736,-1.128485,-1.130071,-1.111905,-0.759029,-1.150206


Feature columns (excluding target): 14


# 4. Train/Test Split
Create a chronological split with the last partition as test data (date-based split).

In [4]:
train_df = model_df[model_df.index < "2025-01-01"]
test_df = model_df[model_df.index >= "2025-01-01"]

feature_cols = [c for c in model_df.columns if c != "target"]
X_train, y_train = train_df[feature_cols], train_df["target"]
X_test, y_test = test_df[feature_cols], test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("Features:", len(feature_cols))

Train size: 1218
Test size: 329
Features: 14


# 5. Time Series Cross Validation
Run baseline cross-validation using `TimeSeriesSplit(n_splits=5)` and report fold RMSE.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)
baseline_rmses = []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    model = LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
        device="gpu"
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

    preds = model.predict(X_val)
    rmse = float(np.sqrt(mean_squared_error(y_val, preds)))
    baseline_rmses.append(rmse)
    print(f"Fold {fold} RMSE: {rmse:.6f}")

print(f"Mean CV RMSE: {np.mean(baseline_rmses):.6f}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.106667
Early stopping, best iteration is:
[147]	valid_0's l2: 0.104814
Fold 1 RMSE: 0.323750
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0512327
[200]	valid_0's l2: 0.0472808
[300]	valid_0's l2: 0.0466374
Early stopping, best iteration is:
[337]	valid_0's l2: 0.0465291
Fold 2 RMSE: 0.215706
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0407856
Early stopping, best iteration is:
[142]	valid_0's l2: 0.0394553
Fold 3 RMSE: 0.198634
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.0330244
[200]	valid_0's l2: 0.0320227
Early stopping, best iteration is:
[245]	valid_0's l2: 0.0319447
Fold 4 RMSE: 0.178731
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 0.116223
Early stopping, best iteration is:
[124]	valid_0's l2: 0.11147
Fold 5 RMSE: 0.333872
Mean CV RMSE: 0.250138


# 6. Hyperparameter Tuning (Optuna)
Tune LightGBM hyperparameters with 20 Optuna trials using time-series CV and GPU execution.

In [6]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "objective": "regression",
        "device": "gpu"
    }

    cv = TimeSeriesSplit(n_splits=3)
    rmses = []
    for tr_idx, val_idx in cv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
        pred = model.predict(X_val)
        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        rmses.append(rmse)

    print(f"Trial {trial.number} | RMSE: {float(np.mean(rmses)):.4f} | params: {trial.params}")
    return float(np.mean(rmses))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print("Best RMSE:", study.best_value)
print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

[I 2026-05-27 18:04:59,297] A new study created in memory with name: no-name-a3597541-f3e7-494b-a7ba-3658df7e7f8e


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[218]	valid_0's l2: 0.107385
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[220]	valid_0's l2: 0.0483884
Training until validation scores don't improve for 50 rounds


[I 2026-05-27 18:05:08,645] Trial 0 finished with value: 0.4193563838616216 and parameters: {'n_estimators': 443, 'learning_rate': 0.05955548766279783, 'max_depth': 6, 'subsample': 0.9337169805585895, 'colsample_bytree': 0.8917238353613364, 'min_child_weight': 5}. Best is trial 0 with value: 0.4193563838616216.


Early stopping, best iteration is:
[193]	valid_0's l2: 0.504666
Trial 0 | RMSE: 0.4194 | params: {'n_estimators': 443, 'learning_rate': 0.05955548766279783, 'max_depth': 6, 'subsample': 0.9337169805585895, 'colsample_bytree': 0.8917238353613364, 'min_child_weight': 5}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[41]	valid_0's l2: 0.104656
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[34]	valid_0's l2: 0.0474542
Training until validation scores don't improve for 50 rounds


[I 2026-05-27 18:05:12,136] Trial 1 finished with value: 0.41647200031591636 and parameters: {'n_estimators': 216, 'learning_rate': 0.19078441653237532, 'max_depth': 8, 'subsample': 0.7323616832211169, 'colsample_bytree': 0.6443978894989768, 'min_child_weight': 2}. Best is trial 1 with value: 0.41647200031591636.


...

Early stopping, best iteration is:
[106]	valid_0's l2: 0.496359
Trial 17 | RMSE: 0.4184 | params: {'n_estimators': 857, 'learning_rate': 0.2539016035837386, 'max_depth': 9, 'subsample': 0.8702987551090473, 'colsample_bytree': 0.7711033663672446, 'min_child_weight': 6}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[32]	valid_0's l2: 0.107468
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[30]	valid_0's l2: 0.0508038
Training until validation scores don't improve for 50 rounds


[I 2026-05-27 18:06:02,292] Trial 18 finished with value: 0.4210216347367332 and parameters: {'n_estimators': 348, 'learning_rate': 0.24416167722035254, 'max_depth': 7, 'subsample': 0.8112452635535926, 'colsample_bytree': 0.9940394786487072, 'min_child_weight': 8}. Best is trial 14 with value: 0.41320831787710643.


Early stopping, best iteration is:
[27]	valid_0's l2: 0.50388
Trial 18 | RMSE: 0.4210 | params: {'n_estimators': 348, 'learning_rate': 0.24416167722035254, 'max_depth': 7, 'subsample': 0.8112452635535926, 'colsample_bytree': 0.9940394786487072, 'min_child_weight': 8}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[149]	valid_0's l2: 0.10779
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[64]	valid_0's l2: 0.0486214
Training until validation scores don't improve for 50 rounds


[I 2026-05-27 18:06:03,718] Trial 19 finished with value: 0.41916882108192466 and parameters: {'n_estimators': 635, 'learning_rate': 0.11565452228276143, 'max_depth': 3, 'subsample': 0.9072462286489076, 'colsample_bytree': 0.8186468757500757, 'min_child_weight': 4}. Best is trial 14 with value: 0.41320831787710643.


Early stopping, best iteration is:
[250]	valid_0's l2: 0.50224
Trial 19 | RMSE: 0.4192 | params: {'n_estimators': 635, 'learning_rate': 0.11565452228276143, 'max_depth': 3, 'subsample': 0.9072462286489076, 'colsample_bytree': 0.8186468757500757, 'min_child_weight': 4}
Best RMSE: 0.41320831787710643
Best params:
  n_estimators: 664
  learning_rate: 0.21964560093337343
  max_depth: 9
  subsample: 0.8721305453069284
  colsample_bytree: 0.7512423130307588
  min_child_weight: 8


# 7. Final Model Training
Train the final model with best Optuna parameters on the full training set and evaluate on the holdout test set.

In [7]:
final_params = dict(best_params)
final_params.update({
    "random_state": 42,
    "objective": "regression",
    "device": "gpu"
})

final_model = LGBMRegressor(**final_params)
final_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

test_pred = final_model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, test_pred)))
mae = float(mean_absolute_error(y_test, test_pred))
mape = float(np.mean(np.abs((y_test - test_pred) / y_test.replace(0, np.nan))) * 100)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.4f}%")

print("Retraining on full dataset...")
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])
final_model.fit(X_full, y_full, callbacks=[lgb.log_evaluation(100)])
print("Final model trained on full dataset")

# Auto-save
model_path = ARTIFACTS_DIR / f"{safe_ticker}_lightgbm_model.pkl"
best_params_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_best_params.json"
feature_cols_path = ARTIFACTS_DIR / f"{safe_ticker}_feature_columns.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print(f"AUTO SAVED as {safe_ticker}_*")
print(f"- Model: {model_path.resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[27]	valid_0's l2: 0.26372
RMSE: 0.513536
MAE:  0.316379
MAPE: 18.0973%
Retraining on full dataset...
Final model trained on full dataset
AUTO SAVED as SBIN_NS_*
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\SBIN_NS_lightgbm_model.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\SBIN_NS_feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\SBIN_NS_lgbm_best_params.json


# 8. Feature Importance
Visualize the top 15 most important predictors from the trained LightGBM model.

In [8]:
importance = pd.Series(final_model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig_imp = go.Figure(
    go.Bar(
        x=importance.values[::-1],
        y=importance.index[::-1],
        orientation="h",
        marker_color="#2563eb"
    )
)
fig_imp.update_layout(
    title=f"{TICKER} LightGBM Top 15 Feature Importance",
    template="plotly_white",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_feature_importance.html"
fig_imp.write_html(str(out_path))
print(f"Wrote feature importance to {out_path.resolve()}")

Wrote feature importance to C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\SBIN_NS_lgbm_feature_importance.html


# 9. Forecast Plot
Plot holdout-period actual vs predicted values and a 30-day future forecast using the latest feature row as a simple forward scenario.

In [9]:
future_steps = 30
last_row = X_test.iloc[[-1]] if len(X_test) > 0 else X_train.iloc[[-1]]
future_X = pd.concat([last_row] * future_steps, ignore_index=True)
future_pred = final_model.predict(future_X)

if isinstance(test_df.index, pd.DatetimeIndex):
    future_idx = pd.bdate_range(start=test_df.index[-1] + pd.tseries.offsets.BDay(1), periods=future_steps)
else:
    future_idx = pd.RangeIndex(start=len(test_df), stop=len(test_df) + future_steps)

y_actual = df.loc[test_df.index, "Close"]
pred_actual = preprocessor.inverse_transform(test_pred)
future_pred_actual = preprocessor.inverse_transform(future_pred)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=test_df.index, y=y_actual, mode="lines", name="Actual", line=dict(color="#111827", width=2)))
fig_fc.add_trace(go.Scatter(x=test_df.index, y=pred_actual, mode="lines", name="Predicted", line=dict(color="#2563eb", width=2)))
fig_fc.add_trace(go.Scatter(x=future_idx, y=future_pred_actual, mode="lines", name="30-Day Forecast", line=dict(color="#10b981", width=2, dash="dash")))
fig_fc.update_layout(
    title=f"{TICKER} Actual vs Predicted + 30-Day Forecast",
    template="plotly_white",
    xaxis_title="Date",
    yaxis_title="Close Price (₹)",
    height=600
)
out_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_forecast.html"
fig_fc.write_html(str(out_path))
print(f"Wrote forecast plot to {out_path.resolve()}")

Wrote forecast plot to C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\SBIN_NS_lgbm_forecast.html


# 10. Save Artifacts
Persist the trained model, preprocessing artifacts, feature columns, and best hyperparameters under the local artifacts directory.

In [10]:
model_path = ARTIFACTS_DIR / f"{safe_ticker}_lightgbm_model.pkl"
feature_cols_path = ARTIFACTS_DIR / f"{safe_ticker}_feature_columns.json"
best_params_path = ARTIFACTS_DIR / f"{safe_ticker}_lgbm_best_params.json"

joblib.dump(final_model, model_path)
preprocessor.save_artifacts()

with feature_cols_path.open("w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with best_params_path.open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)

print("Saved artifacts:")
print(f"- Model: {model_path.resolve()}")
print(f"- Preprocessor scaler: {(ARTIFACTS_DIR / 'scaler.pkl').resolve()}")
print(f"- Feature columns: {feature_cols_path.resolve()}")
print(f"- Best params: {best_params_path.resolve()}")

Saved artifacts:
- Model: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\SBIN_NS_lightgbm_model.pkl
- Preprocessor scaler: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\scaler.pkl
- Feature columns: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\SBIN_NS_feature_columns.json
- Best params: C:\Users\gaura\Desktop\FinSight\notebooks\models\lightgbm\SBIN_NS_lgbm_best_params.json
